In [1]:
import numpy as np
from sklearn.metrics import completeness_score, adjusted_rand_score
import matplotlib.pyplot as plt
from sklearn import cluster
from sklearn.manifold import TSNE
import tensorflow as tf

from tensorflow.keras.datasets import mnist
import numpy as np
from sklearn.metrics import accuracy_score

from sklearn.metrics.pairwise import euclidean_distances
from scipy.spatial.distance import cdist

from sklearn.metrics import accuracy_score
from scipy.spatial.distance import cdist
from scipy.stats import mode

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()            # Load MNIST dataset

#cifar10 = tf.keras.datasets.cifar10                                # Load  CIFAR - 10 dataset
#(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("Training data shape:", x_train.shape, y_train.shape)
print("Test data shape:", x_test.shape, y_test.shape)

y_train = np.squeeze(y_train)                                       # Convert the y array to one-dimensional - reduce the dimension of the y array 
y_test = np.squeeze(y_test)

In [ ]:
x_train_flattened = x_train.reshape(x_train.shape[0], -1)           # Rearrange image data in 2 dimensions
x_test_flattened = x_test.reshape(x_test.shape[0], -1)

x_train_scaled = x_train_flattened.astype('float32') / 255.0        # Normalization for training and test sets
x_test_scaled = x_test_flattened.astype('float32') / 255.0


print("Training data scaler shape:", x_train_scaled.shape, y_train.shape)
print("Test data scaler shape:", x_test_scaled.shape, y_test.shape)

__t-SNE for dimensionality reduction__

The t-SNE method reduces the 784 dimensions to 2.  
This allows the visualization of the data in two dimensions.  
n_components : two-dimensional view  
perplexity : the neighbors taken into account for classifying a sample into a category  
learning_rate : learning rate of the method

In [ ]:
tsne_model = TSNE(n_components=2, perplexity=150, random_state=42)
embedded_data = tsne_model.fit_transform(x_train_scaled)                                # calculate new projection (embedded space) and return 2D data

__Spectral Clustering Application__

Grouping the data into 10 groups  
affinity : how to calculate similarity  
n_clusters : number of groups

In [5]:
clustering_model = cluster.SpectralClustering(
    n_clusters=10, affinity="nearest_neighbors", n_neighbors=10, random_state=42
)
predicted_labels = clustering_model.fit_predict(embedded_data)


__Evaluation of clustering on training data__

ARI (Adjusted Rand Index): Similarity between predicted group and actual category  
Completeness Score: Checking whether points of the same category belong to the same cluster

In [ ]:
ari_train = adjusted_rand_score(y_train, predicted_labels)
completeness_train = completeness_score(y_train, predicted_labels)
print(f"Training ARI: {ari_train:.4f}")
print(f"Training Completeness Score: {completeness_train:.4f}")

__Evaluation on test data__


The groups defined in the cluster are labeled according to the actual labels.

In [ ]:
cluster_to_label = {}                                           # Mapping the clusters to the actual labels
for i in range(10):                                             # For each cluster
    labels_in_cluster = y_train[predicted_labels == i]
    if len(labels_in_cluster) > 0:                              # Check that the cluster is not empty
        cluster_to_label[i] = mode(labels_in_cluster, keepdims=True).mode[0]
    else:
        cluster_to_label[i] = -1  


The centers of each group are calculated from the training data.  
The points belonging to the specific cluster are identified and then their average is calculated.  
The centers are created on which the new data for classification will be based.

In [8]:
cluster_centers = []
for i in range(10):  
    cluster_points = x_train_scaled[predicted_labels == i]
    if len(cluster_points) > 0:
        cluster_centers.append(np.mean(cluster_points, axis=0))
    else:
        cluster_centers.append(np.zeros(x_train_scaled.shape[1]))  
cluster_centers = np.array(cluster_centers) 

Based on the cluster centers calculated earlier, the group that is the shortest possible distance from each center is identified.  
Each point is considered equivalent to the corresponding category.  
For each point assignment to a group, it also labels the sample.  
Finally, the accuracy is calculated and displayed based on the predicted and actual labels.

In [ ]:
distances = cdist(x_test_scaled, cluster_centers, metric="cosine")
test_predicted_clusters = np.argmin(distances, axis=1)

test_predicted_labels = np.array([cluster_to_label[label] for label in test_predicted_clusters])            # We map the test set data to the cluster labels

test_accuracy = accuracy_score(y_test, test_predicted_labels)                                               #Accuracy calculation
print(f"Test Set Accuracy: {test_accuracy:.4f}")

__Visualization of training results__


In [ ]:
# Training data visualization
with plt.style.context("fivethirtyeight"):
    plt.figure(figsize=(10, 8))
    plt.title("t-SNE & Spectral Clustering on MNIST Dataset (Training)")
    scatter_plot = plt.scatter(
        embedded_data[:, 0],
        embedded_data[:, 1],
        c=predicted_labels,
        s=50,
        cmap=plt.cm.get_cmap("jet", 9),
    )
    plt.colorbar(scatter_plot, ticks=range(10))
    plt.clim(-0.5, 9.5)
    plt.show()

ari_train = adjusted_rand_score(y_train, predicted_labels)
completeness_train = completeness_score(y_train, predicted_labels)
print(f"Training ARI: {ari_train:.4f}")
print(f"Training Completeness Score: {completeness_train:.4f}")